<a href="https://colab.research.google.com/github/LucySanders84/genomic-classifier-llm/blob/integrate-from-colab/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',
force_remount=True)

Mounted at /content/drive


In [ ]:
# Remove potentially conflicting packages
!pip -q uninstall -y torch torchvision torchaudio triton transformers accelerate datasets evaluate numpy jax jaxlib

# Upgrade pip
!pip -q install --upgrade pip

# Install PyTorch CUDA build
!pip -q install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.2.2+cu121 torchvision==0.17.2+cu121 torchaudio==2.2.2+cu121

# Install compatible stack
!pip -q install numpy==1.26.4
!pip -q install triton==2.2.0
!pip -q install transformers==4.40.2 accelerate==0.29.3 datasets==2.19.1 evaluate==0.4.2 scikit-learn



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
chex 0.1.90 requires jaxlib>=0.4.27, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
flax 0.11.2 requires jax>=0.6.0, which is not installed.
optax 0.2.6 requires jax>=0.5.3, which is not installed.
optax 0.2.6 requires jaxlib>=0.5.3, which is not installed.
orbax-checkpoint 0.11.31 requires jax>=0.6.0, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is inc

In [ ]:
# Restart runtime to remove cached dependency versions
import os
os.kill(os.getpid(), 9)

In [ ]:
import numpy as np
print("numpy:", np.__version__)
import torch
print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)
import triton
print("triton:", triton.__version__)
import accelerate
print("accelerate:", accelerate.__version__)
import datasets
print("datasets:", datasets.__version__)
import evaluate
print("evaluate:", evaluate.__version__)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

numpy: 1.26.4
torch: 2.2.2+cu121
transformers: 4.40.2
triton: 2.2.0
accelerate: 0.29.3
datasets: 2.19.1
evaluate: 0.4.2
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip -q uninstall -y peft sentence-transformers torchtune

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    set_seed
)

import evaluate
from sklearn.metrics import average_precision_score

%cd /content/drive/MyDrive/dnabert_project

SEED = 42
set_seed(SEED)

/content/drive/MyDrive/dnabert_project


In [ ]:
MODEL = "quietflamingo/dnabert2-fixed"

MAX_LEN = 128
BATCH_TRAIN = 8
BATCH_EVAL = 16
LR = 2e-5
EPOCHS = 2

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [ ]:
def load_csv(path):
    df = pd.read_csv(path)

    if "sequence" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{path} must have columns sequence,label. Found: {list(df.columns)}")

    df["sequence"] = df["sequence"].astype(str).str.replace(" ", "").str.upper()
    df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)

    return Dataset.from_pandas(df, preserve_index=False)

train_ds = load_csv("train.csv")
val_ds = load_csv("val.csv")
test_ds = load_csv("test.csv")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)

def tokenize(batch):
    return tokenizer(
        batch["sequence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["sequence"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["sequence"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["sequence"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/45313 [00:00<?, ? examples/s]

Map:   0%|          | 0/5664 [00:00<?, ? examples/s]

Map:   0%|          | 0/5665 [00:00<?, ? examples/s]

In [ ]:
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]

train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [ ]:
from transformers.models.bert.configuration_bert import BertConfig

class DNABERT2Classifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()

        config = BertConfig.from_pretrained(model_name)

        self.encoder = AutoModel.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True
        )

        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(outputs, tuple):
            last_hidden = outputs[0]
        else:
            last_hidden = outputs.last_hidden_state

        cls_emb = last_hidden[:, 0, :]
        logits = self.classifier(self.dropout(cls_emb))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = DNABERT2Classifier(MODEL, num_labels=2)

config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

bert_layers.py: 0.00B [00:00, ?B/s]

bert_padding.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-fixed:
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-fixed:
- bert_layers.py
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

/root/.cache/huggingface/modules/transformers_modules/quietflamingo/dnabert2-fixed/813031b2bf86d9e960a027e2734c908009f31601/bert_layers.py:123: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at quietflamingo/dnabert2-fixed and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
roc_auc = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = np.array(logits)
    labels = np.array(labels)

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=1)
    pos_probs = probs[:, 1]

    out = {}

    out["accuracy"] = accuracy.compute(
        predictions=preds,
        references=labels
    )["accuracy"]

    out["f1"] = f1.compute(
        predictions=preds,
        references=labels,
        average="binary"
    )["f1"]

    try:
        out["roc_auc"] = roc_auc.compute(
            prediction_scores=pos_probs,
            references=labels
        )["roc_auc"]
    except Exception:
        out["roc_auc"] = float("nan")

    try:
        out["pr_auc"] = average_precision_score(labels, pos_probs)
    except Exception:
        out["pr_auc"] = float("nan")

    return out

In [ ]:
training_args = TrainingArguments(

    output_dir="dnabert2_output",
    seed=SEED,

    learning_rate=LR,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,

    gradient_accumulation_steps=2,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2,

    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Roc Auc,Pr Auc
0,0.034800,0.051243,0.989760,0.989767,0.996337,0.994742
1,0.027000,0.037475,0.992585,0.992590,0.998065,0.996804


TrainOutput(global_step=5664, training_loss=0.04899211331023335, metrics={'train_runtime': 1283.1771, 'train_samples_per_second': 70.626, 'train_steps_per_second': 4.414, 'total_flos': 0.0, 'train_loss': 0.04899211331023335, 'epoch': 1.9996469549867608})

In [ ]:
print("Validation metrics")
print(trainer.evaluate(val_ds))

print("Test metrics")
print(trainer.evaluate(test_ds))

Validation metrics


{'eval_loss': 0.03747468441724777, 'eval_accuracy': 0.9925847457627118, 'eval_f1': 0.992589978828511, 'eval_roc_auc': 0.998065067158759, 'eval_pr_auc': 0.9968038070296157, 'eval_runtime': 18.0879, 'eval_samples_per_second': 313.138, 'eval_steps_per_second': 19.571, 'epoch': 1.9996469549867608}
Test metrics
{'eval_loss': 0.03181886672973633, 'eval_accuracy': 0.9941747572815534, 'eval_f1': 0.9942995335982034, 'eval_roc_auc': 0.9990084670009436, 'eval_pr_auc': 0.9990657508211216, 'eval_runtime': 17.7416, 'eval_samples_per_second': 319.306, 'eval_steps_per_second': 20.009, 'epoch': 1.9996469549867608}


In [ ]:
trainer.save_model("/content/drive/MyDrive/dnabert_project/model")
tokenizer.save_pretrained("/content/drive/MyDrive/dnabert_project/model")

('/content/drive/MyDrive/dnabert_project/model/tokenizer_config.json',
 '/content/drive/MyDrive/dnabert_project/model/special_tokens_map.json',
 '/content/drive/MyDrive/dnabert_project/model/tokenizer.json')